# Remove unnecessary rows

In [3]:
# %% [markdown]
# # add_pre_window_to_matched_pairs.py

# %%
"""
add_pre_window_to_matched_pairs.py

WHAT THIS DOES
--------------
For each of the 10 matched_control_<metric>.csv files (one row = one
treatment/control pair):

  1. DROP the "snapshot" metric columns (e.g. treatment_new_forks,
     treatment_forks, control_new_forks, control_forks). They were only
     measured on the date of collection, so they aren't comparable.

  2. ADD 12 new columns holding the metric's real monthly history for the
     6 months BEFORE the HN submission month (t-0 itself is excluded):

        t-6 ... t-1   -> the TREATMENT repo's values
        c-6 ... c-1   -> the CONTROL repo's values

     The values come from the monthly panels:
        treatment: hn-stories-gh-ai-metrics-cumulative.csv
        control:   nonhn-gh-ai-metrics-cumulative.csv

  "Which metric?" is read from the file name:
      matched_control_new_forks.csv  ->  panel column 'new_forks'
      matched_control_cumulative_stars.csv -> panel column 'cumulative_stars'

KEY DESIGN CHOICE
-----------------
A control repo has no HN submission date of its own. So BOTH t-k and c-k are
anchored on the *treatment* repo's HN submission month (t-0). That is what
"aligned in event time" means in a DiD design. It also means the same
control repo can get DIFFERENT c-k values in different rows (matching is
many-to-one), so we compute the window per PAIR, never per control repo.

MISSING VALUES
--------------
A value is left blank (NaN / <NA>) -- never filled with 0 -- when:
  * the repo is not in the panel file at all, or
  * the target month falls outside the panel (panel starts 2022-05, so an
    HN submission in 2022-06 has no t-6 = 2021-12).
Zero means "we observed no activity"; blank means "we don't know".
A diagnostic summary is printed for each file so nothing is silent.

OUTPUT
------
Originals are NOT touched. New files are written to OUTPUT_DIR with the
same names, so match-unmatch-analysis.py keeps working if you point its
MATCH_DIR at OUTPUT_DIR (it only reads treatment_repo / control_repo).
"""

# %%
import pandas as pd
from pathlib import Path

# ============================================================
# CONFIG -- check these before running
# ============================================================

MATCH_DIR = Path(".")                         # where matched_control_<metric>.csv live
OUTPUT_DIR = Path("matched_control_with_window")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TREATMENT_METRICS_FILE = Path("hn-stories-gh-ai-metrics-[6months-before-after-hn]-cumulative-v2.csv")
CONTROL_METRICS_FILE = Path("nonhn-gh-ai-metrics-cumulative.csv")
TREATMENT_REPO_COL = "repo_full_name"         # repo identifier in the treatment panel
CONTROL_REPO_COL = "control_repo"             # repo identifier in the control panel

PRE_MONTHS = 6                                # t-6 ... t-1

MONTHLY_METRICS = ['new_forks', 'new_stars', 'new_commits', 'new_prs', 'active_contributors']
CUMULATIVE_METRICS = ['cumulative_stars', 'cumulative_forks', 'cumulative_commits',
                      'cumulative_prs', 'cumulative_ve_contributors']
ALL_METRICS = MONTHLY_METRICS + CUMULATIVE_METRICS

# WHITELIST of columns that survive. Everything else in a match file is
# treated as a snapshot metric column and dropped. Using a whitelist (rather
# than a list of columns to drop) means we don't need to know the exact
# metric column names in each of the 10 files.
KEEP_COLS = [
    'treatment_repo', 'treatment_repo_url', 'hn_submission_date',
    'treatment_creation_date', 'control_repo', 'control_creation_date',
    'similarity_score', 'euclidean_distance', 'ranking',
]


# %%
# ============================================================
# 1. LOAD THE MONTHLY PANELS (once, up front)
# ============================================================

def load_panel(path, repo_col):
    """Read a long-format monthly panel and index it by (repo, month).

    `month` is converted to a pandas Period ('2024-03'), which is a real
    calendar-month type: period - 3 means "3 months earlier" and handles
    year boundaries for us (2024-01 minus 2 -> 2023-11).
    """
    df = pd.read_csv(path, parse_dates=['month'])
    df['month'] = df['month'].dt.to_period('M')
    df = df.rename(columns={repo_col: 'repo'})

    # reindex() below needs a UNIQUE index. If a repo had two rows for the
    # same month, lookups would be ambiguous -- fail loudly instead.
    dupes = df.duplicated(['repo', 'month']).sum()
    if dupes:
        raise ValueError(f"{path}: {dupes} duplicate (repo, month) rows")

    return df.set_index(['repo', 'month']).sort_index()


# %%
# ============================================================
# 2. LOOK UP A PRE-HN WINDOW FOR ONE SIDE (treatment OR control)
# ============================================================

def lookup_window(panel, metric, repos, anchor_months, prefix):
    """Return a DataFrame with columns f'{prefix}-6' ... f'{prefix}-1'.

    repos          : Series of repo names, one per pair row
    anchor_months  : Series of Period[M], the HN month (t-0) of each pair row
    prefix         : 't' or 'c'

    HOW THE LOOKUP WORKS (no Python loop over rows):
      For each lag k we build one (repo, month) key per pair row, then ask
      the panel for all of them at once with .reindex(keys). Keys that
      don't exist come back as NaN.
    """
    values = panel[metric]
    out = {}
    for k in range(PRE_MONTHS, 0, -1):           # 6, 5, 4, 3, 2, 1
        keys = pd.MultiIndex.from_arrays([repos.to_numpy(), (anchor_months - k).to_numpy()])
        # reindex gives floats (because of NaN); "Int64" is pandas' nullable
        # integer, so counts stay whole numbers and gaps show as <NA>.
        out[f'{prefix}-{k}'] = values.reindex(keys).to_numpy()
    return pd.DataFrame(out, index=repos.index).astype('Int64')


# %%
# ============================================================
# 3. PROCESS ONE MATCH FILE
# ============================================================

def process_metric(metric, treatment_panel, control_panel):
    in_path = MATCH_DIR / f"matched_control_{metric}.csv"
    if not in_path.exists():
        print(f"[skip] {in_path.name} not found")
        return None

    for name, panel in [('treatment', treatment_panel), ('control', control_panel)]:
        if metric not in panel.columns:
            raise KeyError(f"'{metric}' is not a column in the {name} panel")

    pairs = pd.read_csv(in_path)

    # ---- (1) drop snapshot metric columns via the whitelist ----
    missing_keep = [c for c in KEEP_COLS if c not in pairs.columns]
    if missing_keep:
        raise KeyError(f"{in_path.name} is missing expected columns: {missing_keep}")
    dropped = [c for c in pairs.columns if c not in KEEP_COLS]
    pairs = pairs[KEEP_COLS].copy()

    # ---- (2) HN month = the anchor (t-0) for BOTH sides ----
    # The match file stores a tz-aware string ('2024-03-29 17:37:36+00:00');
    # utc=True parses it safely, tz_localize(None) drops the tz, and
    # to_period('M') truncates to the calendar month.
    hn_month = (pd.to_datetime(pairs['hn_submission_date'], utc=True)
                  .dt.tz_localize(None)
                  .dt.to_period('M'))

    t_cols = lookup_window(treatment_panel, metric, pairs['treatment_repo'], hn_month, 't')
    c_cols = lookup_window(control_panel, metric, pairs['control_repo'], hn_month, 'c')
    result = pd.concat([pairs, t_cols, c_cols], axis=1)

    # ---- (3) diagnostics: never let NaNs be a silent surprise ----
    t_repos_in_panel = set(treatment_panel.index.get_level_values('repo'))
    c_repos_in_panel = set(control_panel.index.get_level_values('repo'))
    t_absent = ~pairs['treatment_repo'].isin(t_repos_in_panel)
    c_absent = ~pairs['control_repo'].isin(c_repos_in_panel)
    t_gaps = t_cols.isna().any(axis=1)
    c_gaps = c_cols.isna().any(axis=1)

    print(f"\n=== {metric}  ({len(pairs)} pairs) ===")
    print(f"  dropped columns : {dropped}")
    print(f"  treatment rows with any blank in t-window : {int(t_gaps.sum())} "
          f"(repo absent from panel: {int(t_absent.sum())})")
    print(f"  control   rows with any blank in c-window : {int(c_gaps.sum())} "
          f"(repo absent from panel: {int(c_absent.sum())})")
    if c_absent.any():
        n_missing = pairs.loc[c_absent, 'control_repo'].nunique()
        print(f"  [!] {n_missing} distinct control repos are missing from "
              f"{CONTROL_METRICS_FILE.name} -> their c-window is blank")

    result.to_csv(OUTPUT_DIR / in_path.name, index=False)
    return result


# %%
# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":
    treatment_panel = load_panel(TREATMENT_METRICS_FILE, TREATMENT_REPO_COL)
    control_panel = load_panel(CONTROL_METRICS_FILE, CONTROL_REPO_COL)

    for metric in ALL_METRICS:
        process_metric(metric, treatment_panel, control_panel)

    print(f"\nDone. Files written to: {OUTPUT_DIR.resolve()}")


=== new_forks  (85 pairs) ===
  dropped columns : ['treatment_new_forks', 'treatment_forks', 'control_new_forks', 'control_forks']
  treatment rows with any blank in t-window : 0 (repo absent from panel: 0)
  control   rows with any blank in c-window : 0 (repo absent from panel: 0)

=== new_stars  (60 pairs) ===
  dropped columns : ['treatment_new_stars', 'treatment_forks', 'control_new_stars', 'control_forks']
  treatment rows with any blank in t-window : 0 (repo absent from panel: 0)
  control   rows with any blank in c-window : 0 (repo absent from panel: 0)

=== new_commits  (37 pairs) ===
  dropped columns : ['treatment_new_commits', 'treatment_forks', 'control_new_commits', 'control_forks']
  treatment rows with any blank in t-window : 0 (repo absent from panel: 0)
  control   rows with any blank in c-window : 0 (repo absent from panel: 0)

=== new_prs  (39 pairs) ===
  dropped columns : ['treatment_new_prs', 'treatment_forks', 'control_new_prs', 'control_forks']
  treatment rows